## Modeling: Customer Churn Prediction

### 1. Feature Engineering
- Loại bỏ `customer_id` (định danh, không mang thông tin dự đoán).
- Xử lý 11 giá trị thiếu ở `total_charges` bằng median imputation (chọn median
  thay vì mean vì cột này có khả năng bị lệch phân phối).
- One-hot encode 11 biến categorical (`contract`, `payment_method`,
  `internet_service`...) với `drop_first=True` để tránh dummy variable trap.
- Kết quả: **30 features** sau encoding, từ 20 cột gốc.

### 2. Train/Test Split
- Tỷ lệ 80/20, dùng `stratify=y` để giữ đúng tỷ lệ churn (26.54%) ở cả 2 tập.
- Train: 5,634 mẫu | Test: 1,409 mẫu.

### 3. Xử lý Class Imbalance
- Tỷ lệ churn/không churn ~1:3 — imbalance mức trung bình, không dùng SMOTE.
- Logistic Regression: `class_weight='balanced'`
- XGBoost: `scale_pos_weight` = tỷ lệ giữa 2 class (tương đương balanced weighting)

### 4. So sánh Model

| Metric            | Logistic Regression | XGBoost |
|---                |---                  |---      |
| ROC-AUC           | 0.843               | 0.816   |
| F1-score (Churn)  | 0.618               | 0.611   |
| Recall (Churn)    | 0.79                | 0.71    |
| Precision (Churn) | 0.51                | 0.54    |
| Accuracy          | 0.74                | 0.76    |

**Quyết định: chọn Logistic Regression làm model chính.**

Lý do:
- AUC và F1 (metric ưu tiên vì bài toán imbalanced) đều cao hơn XGBoost.
- XGBoost dùng tham số mặc định, chưa tuning — với dataset tương đối nhỏ (7,043
  dòng) và các mối quan hệ giữa feature/churn khá tuyến tính (thấy rõ từ EDA),
  model đơn giản hơn generalize tốt hơn là điều hợp lý, không phải bất thường.
- Logistic Regression còn có ưu điểm dễ diễn giải (interpretable) — quan trọng
  khi cần giải thích quyết định model cho stakeholder không chuyên kỹ thuật.

### 5. Trade-off Precision vs Recall (ý nghĩa business)

Model có recall cao (0.79) nhưng precision thấp hơn (0.51) đối với nhóm Churn —
nghĩa là model ưu tiên **không bỏ sót khách hàng có nguy cơ rời bỏ**, chấp nhận
một số cảnh báo nhầm. Trong bài toán churn, đây là lựa chọn hợp lý: bỏ sót một
khách hàng thực sự sắp rời bỏ (false negative) thường tốn kém hơn nhiều so với
gửi nhầm ưu đãi giữ chân cho khách hàng không có ý định rời bỏ (false positive).

### 6. Feature Importance (Logistic Regression coefficients)

| Feature                        | Coefficient | Diễn giải                                                              |
|---                             |---          |-- -                                                                    |
| `tenure`                       | -1.227      | Ảnh hưởng mạnh nhất. Gắn bó lâu → churn giảm rõ rệt.                   |
| `monthly_charges`              | -0.841      | Phí hàng tháng cao hơn → churn thấp hơn (cần xem cùng lưu ý ở mục 7).  |
| `internet_service_Fiber optic` | +0.712      | Dùng Fiber optic → tăng khả năng churn.                                |
| `contract_Two year`            | -0.659      | Hợp đồng 2 năm → giảm mạnh churn.                                      |
| `total_charges`                | +0.590      | Cần diễn giải thận trọng (xem mục 7).                                  |
| `contract_One year`            | -0.318      | Giảm churn, nhưng ít hơn Two year — đúng logic bậc thang theo cam kết. |

**Đối chiếu với EDA:** Các feature quan trọng nhất theo model (`tenure`,
`contract`, `internet_service`) hoàn toàn khớp với insight tìm được bằng SQL ở
bước EDA — hai phương pháp độc lập (thống kê mô tả và machine learning) cho
kết luận nhất quán, củng cố độ tin cậy của phát hiện.

### 7. Limitations

- **Đa cộng tuyến (multicollinearity):** `total_charges` có tương quan cao với
  `tenure × monthly_charges` (vì về bản chất là tích lũy của 2 biến này). Khi
  cả 3 biến cùng vào model tuyến tính, coefficient riêng lẻ có thể bị biến dạng
  — vì vậy dấu dương của `total_charges` không nên diễn giải trực tiếp là
  "tổng chi cao → dễ churn" mà cần thêm phân tích (VIF hoặc bỏ bớt 1 biến
  tương quan) nếu muốn kết luận chắc chắn hơn.
- **XGBoost chưa được tuning hyperparameters** — kết quả so sánh hiện tại chưa
  hoàn toàn công bằng; nếu có thêm thời gian, nên thử GridSearch/RandomizedSearch
  cho `max_depth`, `n_estimators`, `learning_rate` trước khi kết luận cuối cùng.
- Dataset là snapshot tại 1 thời điểm (không có yếu tố thời gian/seasonality),
  nên model chưa phản ánh được xu hướng churn thay đổi theo mùa hoặc theo thời
  gian dài hạn.

### 8. Next Steps
- Đóng gói model vào FastAPI endpoint `/predict` để phục vụ dự đoán real-time.
- (Nếu có thời gian) Thử SHAP values để giải thích prediction ở cấp độ từng
  khách hàng cụ thể, thay vì chỉ coefficient tổng quát.